<a href="https://colab.research.google.com/github/1pe1pe/TSMC-Stock-Tracker/blob/main/0052_tracker_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install yfinance FinMind
import yfinance as yf
from FinMind.data import DataLoader
from datetime import datetime, timedelta

def get_0052_data():
    print("--- 晨間 富邦科技 (0052) 數據彙整 ---")
    print(f"執行日期: {datetime.now().strftime('%Y-%m-%d')}\n")

    # ================= 1. 股價與技術面 (yfinance) =================
    print("📍 股價與技術面:")
    try:
        # 往前抓 40 天資料，確保有足夠天數算出精準的 20 日均線(月線)
        stock_tw = yf.Ticker("0052.TW")
        hist_tw = stock_tw.history(period="40d")

        if not hist_tw.empty:
            # 計算均線 (5日為周線，20日為月線)
            hist_tw['5MA'] = hist_tw['Close'].rolling(window=5).mean()
            hist_tw['20MA'] = hist_tw['Close'].rolling(window=20).mean()

            latest_close = hist_tw['Close'].iloc[-1]
            latest_5ma = hist_tw['5MA'].iloc[-1]
            latest_20ma = hist_tw['20MA'].iloc[-1]

            print(f"【最新收盤與均線】")
            print(f"- 最新收盤價: {latest_close:.2f} 元")
            print(f"- 周線 (5MA) : {latest_5ma:.2f} 元")
            print(f"- 月線 (20MA): {latest_20ma:.2f} 元\n")

            print("【近 7 日收盤價】")
            last_7_days = hist_tw.tail(7)
            for index, row in last_7_days.iterrows():
                date_str = index.strftime('%Y-%m-%d')
                print(f"- {date_str} : {row['Close']:.2f} 元")

    except Exception as e:
        print(f"股價讀取失敗: {e}")

    # ================= 2. 籌碼面 (FinMind) =================
    print("\n📍 法人與主力籌碼動態 (近 7 個交易日):")
    dl = DataLoader()
    start_date = (datetime.now() - timedelta(days=30)).strftime('%Y-%m-%d')

    # 外資買賣超
    print("\n【外資買賣超張數】")
    try:
        df_inst = dl.taiwan_stock_institutional_investors(stock_id='0052', start_date=start_date)
        if not df_inst.empty:
            foreign_data = df_inst[df_inst['name'] == 'Foreign_Investor']
            if not foreign_data.empty:
                last_7_foreign = foreign_data.tail(7)
                for index, row in last_7_foreign.iterrows():
                    net_buy_sell = (row['buy'] - row['sell']) / 1000
                    date_inst = row['date']
                    trend = "🔴 買超" if net_buy_sell > 0 else "🟢 賣超"
                    print(f"- {date_inst} : {net_buy_sell:+,.0f} 張 ({trend})")
            else:
                print("- 近期無外資交易紀錄")
        else:
            print("- 伺服器目前無外資資料")
    except Exception as e:
        print("- 外資資料抓取失敗 (伺服器連線限制，請稍後重試)")

    # 散戶融資餘額
    print("\n【散戶融資餘額張數】")
    try:
        df_margin = dl.taiwan_stock_margin_purchase_short_sale(stock_id='0052', start_date=start_date)
        if not df_margin.empty:
            last_7_margin = df_margin.tail(7)
            for index, row in last_7_margin.iterrows():
                margin_bal = row['MarginPurchaseBalance'] / 1000
                date_margin = row['date']
                print(f"- {date_margin} : {margin_bal:,.0f} 張")
        else:
             print("- 伺服器目前無融資資料")
    except Exception as e:
        print("- 融資資料抓取失敗 (伺服器連線限制，請稍後重試)")

    print("\n💡 決策提醒：")
    print("1. 股價若站在「周線」與「月線」之上，代表短中線趨勢偏多。")
    print("2. 搭配觀察同日的台積電 (2330) 籌碼，可以更精準抓到 0052 的大方向！")

if __name__ == "__main__":
    get_0052_data()

--- 晨間 富邦科技 (0052) 數據彙整 ---
執行日期: 2026-02-21

📍 股價與技術面:
【最新收盤與均線】
- 最新收盤價: 46.07 元
- 周線 (5MA) : 44.09 元
- 月線 (20MA): 43.16 元

【近 7 日收盤價】
- 2026-02-03 : 43.37 元
- 2026-02-04 : 43.43 元
- 2026-02-05 : 42.86 元
- 2026-02-06 : 42.71 元
- 2026-02-09 : 43.88 元
- 2026-02-10 : 44.95 元
- 2026-02-11 : 46.07 元

📍 法人與主力籌碼動態 (近 7 個交易日):


2026-02-21 10:31:34.032 | INFO     | FinMind.data.finmind_api:login_by_token:84 - Login success
2026-02-21 10:31:34.033 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 0052



【外資買賣超張數】


2026-02-21 10:31:34.970 | INFO     | FinMind.data.finmind_api:get_data:153 - download Dataset.TaiwanStockMarginPurchaseShortSale, data_id: 0052


- 2026-02-03 : -485 張 (🟢 賣超)
- 2026-02-04 : +694 張 (🔴 買超)
- 2026-02-05 : -501 張 (🟢 賣超)
- 2026-02-06 : +280 張 (🔴 買超)
- 2026-02-09 : +1,500 張 (🔴 買超)
- 2026-02-10 : +5,124 張 (🔴 買超)
- 2026-02-11 : +1,414 張 (🔴 買超)

【散戶融資餘額張數】
- 融資資料抓取失敗 (伺服器連線限制，請稍後重試)

💡 決策提醒：
1. 股價若站在「周線」與「月線」之上，代表短中線趨勢偏多。
2. 搭配觀察同日的台積電 (2330) 籌碼，可以更精準抓到 0052 的大方向！
